In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install(
    'transformers<5.0.0',
    'timm>=0.9.12',
    'scikit-learn',
    'torchmetrics',
)
print('All packages installed.')


In [ ]:
#we need to restart the kernel, after upgrade protobuf
%pip install --upgrade protobuf
import google.protobuf
print(google.protobuf.__version__)


In [ ]:
%pip install torch>=2.6.0+cu124

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('causal-conv1d>=1.4.0', 'mamba-ssm>=2.2.0', '--no-build-isolation')

In [ ]:
#! pip install mambavision==1.1.0

In [ ]:
%pip install evaluate

In [1]:
import os, json, time, copy, warnings, random
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image

import timm
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy, LabelSmoothingCrossEntropy

from transformers import AutoModelForImageClassification

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)
from torchmetrics import (
    Accuracy, Precision, Recall, F1Score,
    AUROC, AveragePrecision, CohenKappa, MatthewsCorrCoef
)

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# ── Determinism ───────────────────────────────────────────────────────────────
SEED = 42
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                       'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f' Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')



 Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB


In [ ]:
#!pip3 install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [ ]:
#!pip install torch --index-url https://lnkd.in/g_8dBghS

# 1. Clear out any previous partial or broken installations
#!pip uninstall -y mamba-ssm causal-conv1d

# 2. Install the required causal-conv1d dependency without build isolation
#!pip install causal-conv1d>=1.4.0 --no-build-isolation

# 3. Install the core mamba package without build isolation
#!pip install mamba-ssm --no-build-isolation

In [2]:
from mamba_ssm import Mamba
print("✅ Mamba loaded successfully")

✅ Mamba loaded successfully


In [4]:
import math
import torch
import torch.nn as nn
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput

# Import Mamba block component natively
from mamba_ssm import Mamba

# =====================================================================
# 1. DEFINE THE HYBRID MAMBA-TRANSFORMER BLOCK LAYER
# =====================================================================
class HybridBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim, dropout=0.1):
        super().__init__()

        # Mamba State-Space Module Layer Setup
        self.mamba = Mamba(
            d_model=embed_dim,    # Model dimension
            d_state=16,           # SSM state expansion factor
            d_conv=4,             # Local convolution width
            expand=2              # Block expansion factor
        )
        self.norm1 = nn.LayerNorm(embed_dim)

        # Standard Transformer Multi-Head Self-Attention Layer Setup
        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm2 = nn.LayerNorm(embed_dim)

        # Feed-Forward Network Footprint
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout)
        )
        self.norm3 = nn.LayerNorm(embed_dim)

    def forward(self, x, key_padding_mask=None):
        # 1. Step One: Pass token streams through Mamba block (Linear Complexity)
        # Mamba natively acts over batch_first sequences [Batch, Seq_len, Dim]
        x = x + self.mamba(self.norm1(x))

        # 2. Step Two: Pass token streams through standard multi-head attention
        # PyTorch attention expects an inverted attention mask shape or Key Padding parameters
        attn_out, _ = self.attention(
            self.norm2(x), self.norm2(x), self.norm2(x),
            key_padding_mask=key_padding_mask
        )
        x = x + attn_out

        # 3. Step Three: Pass token streams through FFN block
        x = x + self.ffn(self.norm3(x))
        return x

# =====================================================================
# 2. HYBRID CLASSIFIER TOPOLOGY
# =====================================================================
class HybridMambaTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_classes=2, max_len=512, dropout=0.1):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_len, embed_dim))
        self.dropout = nn.Dropout(dropout)

        # Interleave multiple stacks of the hybrid block architecture
        self.layers = nn.ModuleList([
            HybridBlock(embed_dim, num_heads, hidden_dim, dropout)
            for _ in range(num_layers)
        ])

        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x, attention_mask=None):
        seq_len = x.size(1)

        # Set up positional weights
        pe = self.positional_encoding[:, :seq_len, :]
        out = self.dropout(self.embedding(x) + pe)

        # Map out padding arrays
        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = (attention_mask == 0)

        # Push through stacked hybrid blocks
        for layer in self.layers:
            out = layer(out, key_padding_mask=key_padding_mask)

        # Target [CLS] token context vector mapping layout
        out = out[:, 0, :]
        return self.fc(out)

# =====================================================================
# 3. COMPATIBLE TRAINER ENVELOPE
# =====================================================================
class HFCompatibleWrapper(nn.Module):
    def __init__(self, custom_model):
        super().__init__()
        self.model = custom_model
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask=None, labels=None):
        logits = self.model(input_ids, attention_mask=attention_mask)
        loss = None
        if labels is not None:
            loss = self.criterion(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)

# =====================================================================
# 4. TRANSITION TO PROCESSING FULL 50,000 DATASET
# =====================================================================
print("Loading all 50,000 reviews from IMDb dataset...")
raw_datasets = load_dataset("stanfordnlp/imdb")

# Split the original 25k test set 50/50 into distinct validation and test sets
split_test = raw_datasets["test"].train_test_split(test_size=0.5, seed=42)
raw_datasets["validation"] = split_test["train"]  # 12,500 reviews
raw_datasets["test"] = split_test["test"]          # 12,500 reviews

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

print("Tokenizing the entire dataset...")
# Full allocation scale mapping profile across all dataset partitions
train_dataset = raw_datasets["train"].map(tokenize_function, batched=True)
val_dataset = raw_datasets["validation"].map(tokenize_function, batched=True)
test_dataset = raw_datasets["test"].map(tokenize_function, batched=True)

# Set up data formatting for PyTorch environments
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =====================================================================
# 5. EXECUTE OPTIMIZED SCALED TRAINING LOOP
# =====================================================================
raw_hybrid_model = HybridMambaTransformerClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=256,
    num_heads=8,
    hidden_dim=512,
    num_layers=4,
    dropout=0.2 #0.1
)
model = HFCompatibleWrapper(raw_hybrid_model)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./hybrid_mamba_results",
    learning_rate=5e-5,          # Stable learning rate profile for scaling to large raw tokens
    per_device_train_batch_size=32, # Increased batch size for processing high data density efficiently
    per_device_eval_batch_size=32,
    num_train_epochs=15, #8,          # 3 full epochs across 25k samples provides deep convergence patterns
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./hybrid_logs",
    logging_steps=100,
    fp16=torch.cuda.is_available(), # Essential parameter to avoid hardware out-of-memory errors
    dataloader_num_workers=2     # Multi-threaded background streaming configurations
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting hybrid Mamba-Transformer training across the full dataset...")
trainer.train()

print("\n--- Running evaluation on the final, unseen 12.5k test set ---")
test_results = trainer.evaluate(eval_dataset=test_dataset)
print(f"Final Hybrid Model Test Accuracy: {test_results['eval_accuracy']:.4f}")


Loading all 50,000 reviews from IMDb dataset...
Tokenizing the entire dataset...
Starting hybrid Mamba-Transformer training across the full dataset...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.620400,0.531680,0.733680
2,0.530400,0.467652,0.773760
3,0.471200,0.463309,0.785440
4,0.446200,0.419487,0.812640
5,0.416000,0.426512,0.814640
6,0.391600,0.400197,0.827520
7,0.364300,0.389914,0.831280
8,0.344200,0.390822,0.837360
9,0.319900,0.388942,0.840720
10,0.314600,0.386754,0.843680



--- Running evaluation on the final, unseen 12.5k test set ---


Final Hybrid Model Test Accuracy: 0.8414


In [6]:
import torch
import torch.nn as nn
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput

# Import the native Mamba module
from mamba_ssm import Mamba

# =====================================================================
# 1. PURE MAMBA CLASSIFIER ARCHITECTURE
# =====================================================================
class PureMambaClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_layers, num_classes=2, max_len=512, dropout=0.1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Stacked native Mamba layers with individual normalization steps
        self.layers = nn.ModuleList([
            Mamba(
                d_model=embed_dim,    # Model dimension
                d_state=16,           # SSM state expansion factor
                d_conv=4,             # Local convolution width
                expand=2              # Block expansion factor
            )
            for _ in range(num_layers)
        ])
        
        self.norms = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(num_layers)])
        
        # Simple Linear layer for binary classification output
        self.fc = nn.Linear(embed_dim, num_classes)
        
    def forward(self, x, attention_mask=None):
        # x shape: [batch_size, seq_len]
        out = self.dropout(self.embedding(x))
        
        # Pass sequentially through the stacked Mamba blocks
        for mamba_layer, norm_layer in zip(self.layers, self.norms):
            # Mamba handles sequential context natively along the sequence dimension
            out = out + mamba_layer(norm_layer(out))
            
        # Sequence Pooling: Mean pooling across the sequence dimension (dim=1)
        # We use mean pooling over the sequence mask since pure Mamba does not use a [CLS] token
        if attention_mask is not None:
            # Mask out padding tokens to ensure accurate average vectors
            mask = attention_mask.unsqueeze(-1).float() # [batch_size, seq_len, 1]
            out = out * mask
            pooled_out = out.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            pooled_out = out.mean(dim=1)
            
        return self.fc(pooled_out)

# =====================================================================
# 2. HUGGING FACE COMPATIBLE TRAINER WRAPPER
# =====================================================================
class HFCompatibleWrapper(nn.Module):
    def __init__(self, custom_model):
        super().__init__()
        self.model = custom_model
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask=None, labels=None):
        logits = self.model(input_ids, attention_mask=attention_mask)
        loss = None
        if labels is not None:
            loss = self.criterion(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)

# =====================================================================
# 3. DATASET LIFECYCLE AND PARSING ON ALL 50,000 REVIEWS
# =====================================================================
print("Loading all 50,000 reviews from IMDb dataset...")
raw_datasets = load_dataset("stanfordnlp/imdb")

split_test = raw_datasets["test"].train_test_split(test_size=0.5, seed=42)
raw_datasets["validation"] = split_test["train"]  # 12,500 validation rows
raw_datasets["test"] = split_test["test"]          # 12,500 testing rows

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

print("Tokenizing the entire dataset...")
train_dataset = raw_datasets["train"].map(tokenize_function, batched=True)
val_dataset = raw_datasets["validation"].map(tokenize_function, batched=True)
test_dataset = raw_datasets["test"].map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =====================================================================
# 4. MODEL CONFIGURATION AND EXECUTION RUN PIPELINE
# =====================================================================
raw_mamba_model = PureMambaClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=256,
    num_layers=6,            # Stacked layer depth
    dropout=0.1
)
model = HFCompatibleWrapper(raw_mamba_model)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./pure_mamba_results",
    learning_rate=5e-5,          
    per_device_train_batch_size=32, 
    per_device_eval_batch_size=32,
    num_train_epochs=15,          
    weight_decay=0.01,
    eval_strategy="epoch",       
    save_strategy="epoch",
    load_best_model_at_end=True, 
    metric_for_best_model="accuracy",
    logging_dir="./mamba_logs",
    logging_steps=100,
    fp16=torch.cuda.is_available(), 
    dataloader_num_workers=2     
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,    
    compute_metrics=compute_metrics,
)

print("Starting pure Mamba training loop across full dataset...")
trainer.train()

print("\n--- Running evaluation on the final, unseen 12.5k test set ---")
test_results = trainer.evaluate(eval_dataset=test_dataset)
print(f"Final Pure Mamba Test Accuracy: {test_results['eval_accuracy']:.4f}")


Loading all 50,000 reviews from IMDb dataset...
Tokenizing the entire dataset...
Starting pure Mamba training loop across full dataset...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.582600,0.484735,0.765120
2,0.420400,0.421967,0.805040
3,0.335700,0.362900,0.844320
4,0.300500,0.333567,0.861440
5,0.253700,0.341350,0.863360
6,0.229200,0.340709,0.868960
7,0.214700,0.329719,0.875040
8,0.186900,0.397768,0.859920
9,0.171500,0.376856,0.866560
10,0.158200,0.379522,0.873200



--- Running evaluation on the final, unseen 12.5k test set ---


Final Pure Mamba Test Accuracy: 0.8785


In [8]:
import math
import torch
import torch.nn as nn
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, Trainer, TrainingArguments
from transformers.modeling_outputs import SequenceClassifierOutput

# Import Mamba block components natively
from mamba_ssm import Mamba

# =====================================================================
# 1. CONFIGURABLE HYBRID BLOCK FOOTPRINT
# =====================================================================
class ConfigurableHybridBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim, mamba_to_attn_ratio=3, dropout=0.1):
        super().__init__()
        
        # Setup 'N' successive Mamba layers based on the configured ratio
        self.mamba_layers = nn.ModuleList([
            Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
            for _ in range(mamba_to_attn_ratio)
        ])
        self.mamba_norms = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(mamba_to_attn_ratio)])
        
        # Setup exactly 1 Multi-Head Attention layer at the tail of the block
        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim, 
            num_heads=num_heads, 
            dropout=dropout, 
            batch_first=True
        )
        self.attn_norm = nn.LayerNorm(embed_dim)
        
        # Feed-Forward Network Block
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout)
        )
        self.ffn_norm = nn.LayerNorm(embed_dim)

    def forward(self, x, key_padding_mask=None):
        # 1. Process through N sequential Mamba layers
        for mamba_layer, norm_layer in zip(self.mamba_layers, self.mamba_norms):
            x = x + mamba_layer(norm_layer(x))
            
        # 2. Complete a single global Attention validation step
        attn_out, _ = self.attention(
            self.attn_norm(x), self.attn_norm(x), self.attn_norm(x), 
            key_padding_mask=key_padding_mask
        )
        x = x + attn_out
        
        # 3. Final Multi-Layer Perceptron refinement pass
        x = x + self.ffn(self.ffn_norm(x))
        return x

# =====================================================================
# 2. HYBRID CLASSIFIER TOPOLOGY
# =====================================================================
class ConfigurableHybridClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers=4, mamba_to_attn_ratio=3, num_classes=2, max_len=512, dropout=0.1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_len, embed_dim))
        self.dropout = nn.Dropout(dropout)
        
        # Stack multiple hybrid building blocks (each block contains N Mamba layers and 1 Attention layer)
        self.layers = nn.ModuleList([
            ConfigurableHybridBlock(embed_dim, num_heads, hidden_dim, mamba_to_attn_ratio, dropout) 
            for _ in range(num_layers)
        ])
        
        self.fc = nn.Linear(embed_dim, num_classes)
        
    def forward(self, x, attention_mask=None):
        seq_len = x.size(1)
        pe = self.positional_encoding[:, :seq_len, :]
        out = self.dropout(self.embedding(x) + pe)
        
        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = (attention_mask == 0)
            
        for layer in self.layers:
            out = layer(out, key_padding_mask=key_padding_mask)
            
        # [CLS] token pooling step
        out = out[:, 0, :]
        return self.fc(out)

# =====================================================================
# 3. HUGGING FACE COMPATIBLE TRAINER WRAPPER
# =====================================================================
class HFCompatibleWrapper(nn.Module):
    def __init__(self, custom_model):
        super().__init__()
        self.model = custom_model
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask=None, labels=None):
        logits = self.model(input_ids, attention_mask=attention_mask)
        loss = None
        if labels is not None:
            loss = self.criterion(logits, labels)
        return SequenceClassifierOutput(loss=loss, logits=logits)

# =====================================================================
# 4. DATASET LIFECYCLE AND PARSING ON ALL 50,000 REVIEWS
# =====================================================================
print("Loading all 50,000 reviews from IMDb dataset...")
raw_datasets = load_dataset("stanfordnlp/imdb")

split_test = raw_datasets["test"].train_test_split(test_size=0.5, seed=42)
raw_datasets["validation"] = split_test["train"]  # 12,500 validation rows
raw_datasets["test"] = split_test["test"]          # 12,500 testing rows

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

print("Tokenizing the entire dataset...")
train_dataset = raw_datasets["train"].map(tokenize_function, batched=True)
val_dataset = raw_datasets["validation"].map(tokenize_function, batched=True)
test_dataset = raw_datasets["test"].map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# =====================================================================
# 5. MODEL CONFIGURATION AND EXECUTION RUN PIPELINE
# =====================================================================
raw_hybrid_model = ConfigurableHybridClassifier(
    vocab_size=tokenizer.vocab_size,
    embed_dim=256,
    num_heads=8,
    hidden_dim=512,
    num_layers=4,            # 4 total blocks
    mamba_to_attn_ratio=3,   # DEFAULT: 3 Mamba layers to 1 Attention layer per block
    dropout=0.2 #0.1
)
model = HFCompatibleWrapper(raw_hybrid_model)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./configurable_hybrid_results",
    learning_rate=5e-5,          
    per_device_train_batch_size=32, 
    per_device_eval_batch_size=32,
    num_train_epochs=40,          
    weight_decay=0.01,
    eval_strategy="epoch",       
    save_strategy="epoch",
    load_best_model_at_end=True, 
    metric_for_best_model="accuracy",
    logging_dir="./hybrid_logs",
    logging_steps=100,
    fp16=torch.cuda.is_available(), 
    dataloader_num_workers=2     
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,    
    compute_metrics=compute_metrics,
)

print("Starting configurable hybrid Mamba-Transformer training loop...")
trainer.train()

print("\n--- Running evaluation on the final, unseen 12.5k test set ---")
test_results = trainer.evaluate(eval_dataset=test_dataset)
print(f"Final Hybrid Model Test Accuracy: {test_results['eval_accuracy']:.4f}")


Loading all 50,000 reviews from IMDb dataset...
Tokenizing the entire dataset...
Starting configurable hybrid Mamba-Transformer training loop...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.604700,0.531702,0.741600
2,0.504800,0.501130,0.764320
3,0.451400,0.486593,0.772400
4,0.426300,0.430364,0.812000
5,0.390700,0.433849,0.821440
6,0.369300,0.428304,0.828160
7,0.326600,0.399098,0.843360
8,0.272800,0.403144,0.844640
9,0.263600,0.411429,0.847040
10,0.247300,0.434651,0.847360



--- Running evaluation on the final, unseen 12.5k test set ---


Final Hybrid Model Test Accuracy: 0.8522


In [ ]:
from mamba_ssm import Mamba
print("✅ Mamba loaded successfully")